# ProGen2 Protein Sequence Generation and Scoring

![ProGen2 Protein Sequence Generation and Scoring](https://proto-bio.github.io/proto-assets/images/tool/progen2/hero.png)

This notebook demonstrates both functions of the ProGen2 tool. In the first section, we use `run_progen2_sample` to generate protein sequences from amino acid prompts using autoregressive sampling. In the second section, we use `run_progen2_score` to compute log-likelihood and perplexity metrics that quantify how natural a given sequence appears to the model, enabling ranking and filtering of design candidates.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("progen2")
display_overview("progen2")
display_docs_section("progen2", "Background")

# ProGen2

ProGen2 is an autoregressive protein language model from Salesforce Research, first released in 2022 and published in 2023, trained on natural protein sequences from genomic, metagenomic, and immune-repertoire databases.

ProGen2 ([Nijkamp et al., 2023](https://doi.org/10.1016/j.cels.2023.10.002)) is a family of autoregressive protein language models trained with a next-token prediction objective: during training the model learns to predict the next residue given all preceding residues. The family spans `progen2-small` (151 million parameters) up to `progen2-xlarge` (6.4 billion parameters). The checkpoints were trained on different protein collections as a result of the paper's finding that the training-data distribution has a large and sometimes counterintuitive effect on downstream performance. Most checkpoints are trained on natural proteins drawn from [UniRef90](https://www.uniprot.org/help/uniref) and the [BFD](https://bfd.mmseqs.com/) metagenomic set; `progen2-BFD90` uses the BFD90 collection, and `progen2-oas` is trained on antibody sequences from the [Observed Antibody Space](https://opig.stats.ox.ac.uk/webapps/oas/) database.

The autoregressive training objective instills two primary capabilities. First, new candidate protein sequences can be sampled from a starting prompt via the predicted next-residue distributions. Second, the model can be used to score existing protein sequences, as the likelihood the model assigns to a sequence is shown in the paper to provide a proxy zero-shot fitness score or measure of plausibility with no additional task-specific training.

## Available tools

In [2]:
display_available_tools("progen2")

- **`run_progen2_sample()`** — Sample protein sequences using ProGen2 language model
- **`run_progen2_score()`** — Score protein sequences using ProGen2 language model

### `run_progen2_sample`

ProGen2 generates protein sequences autoregressively from a prompt, extending the sequence one residue at a time according to its learned distribution over natural proteins. Available checkpoints range from 151M to 6B parameters, with a specialized OAS variant for antibody sequences. The `temperature` and `top_p` parameters control sampling diversity: lower temperature produces more conservative, high-confidence extensions while higher values explore more of sequence space.

In [3]:
from proto_tools import (
    ProGen2SampleInput,
    ProGen2SampleConfig,
    run_progen2_sample,
)

In [4]:
# Display docs
display_api_reference("progen2", "input", "run_progen2_sample")

# Create a ProGen2 input with an N-terminal amino acid prompt
inputs = ProGen2SampleInput(prompts=["MKTAYIAKQRQISF"])

**Input** — `CausalModelSampleInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>prompts</code> | <code>list[str]</code> | required | Prompt sequence(s) to condition generation on |

In [5]:
# Display config docs
display_api_reference("progen2", "config", "run_progen2_sample")

# Configure sampling parameters | see docs above for all fields
config = ProGen2SampleConfig(
    model_checkpoint="progen2-large",
    max_new_tokens=120,
    temperature=0.8,
    top_p=0.95,
    top_k=0,
    truncate_at_stop=True,
    strip_special_tokens=True,
    device="cuda",  # Change to "cpu" if no GPU available
)

**Config** — `ProGen2SampleConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run the model on |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>prepend_prompt</code> | <code>bool</code> | <code>True</code> | Include the input prompt at the start of each generated sequence |
| <code>temperature</code> | <code>float</code> | <code>0.2</code> | Softmax temperature for sampling; lower is more deterministic |
| <code>top_p</code> | <code>float</code> | <code>0.95</code> | Nucleus sampling threshold over per-position token probabilities |
| <code>batch_size</code> | <code>int</code> | <code>1</code> | Prompts per GPU forward pass; raise for throughput, lower if OOM |
| <code>model_checkpoint</code> | <code>Literal['progen2-small', 'progen2-medium', 'progen2-base', 'progen2-oas', 'progen2-large', 'progen2-BFD90', 'progen2-xlarge']</code> | <code>'progen2-large'</code> | ProGen2 weights variant |
| <code>local_path</code> | <code>str &#124; None</code> | <code>None</code> | Override the default download with a local weights directory |
| <code>top_k</code> | <code>int</code> | <code>0</code> | Top-k truncation; 0 disables and uses top-p only |
| <code>max_new_tokens</code> | <code>int</code> | <code>256</code> | Maximum newly-generated tokens per prompt (excludes the prompt) |
| <code>truncate_at_stop</code> | <code>bool</code> | <code>True</code> | Truncate generated sequences at the first stop token |
| <code>strip_special_tokens</code> | <code>bool</code> | <code>True</code> | Strip ProGen2 start/stop sentinel tokens from output |
| <code>return_logits</code> | <code>bool</code> | <code>False</code> | Include per-position logits in the output (large; disable to save memory) |

In [6]:
# Run the sampling function
sample_result = run_progen2_sample(inputs, config)

Running run_progen2_sample [00:00]

In [7]:
# Display output docs
display_api_reference("progen2", "output", "run_progen2_sample")

# Show the prompt and the generated continuation
prompt = inputs.prompts[0]
generated = sample_result.sequences[0]
print(f"Prompt:       {prompt}")
print(f"Generated:    {generated[:80]}..." if len(generated) > 80 else f"Generated:    {generated}")
print(f"Total length: {len(generated)} residues")

**Output** — `ProGen2SampleOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequences</code> | <code>list[str]</code> | required | Generated sequences |
| <code>logits</code> | <code>list[list[list[float]]] &#124; None</code> | <code>None</code> | Per-position logits for each generated sequence |

Prompt:       MKTAYIAKQRQISF
Generated:    MKTAYIAKQRQISFGDYEGEINKFLDTNYNLKGFYTLIGNQIISGSQLVKTSVLPPVTTNFQLLPDDLNIANVNLQQKNL...
Total length: 134 residues


### `run_progen2_score`

ProGen2 scores sequences by computing the autoregressive log-likelihood at each position, measuring how predictable each residue is given its preceding context. The resulting perplexity summarizes sequence naturalness: lower perplexity indicates the sequence more closely matches the statistical patterns of natural proteins in the training data. This is useful for ranking designed candidates, filtering out sequences unlikely to fold or function, or comparing wild-type and mutant variants.

In [8]:
from proto_tools import (
    ProGen2ScoringInput,
    ProGen2ScoringConfig,
    run_progen2_score,
)

In [9]:
# Display docs
display_api_reference("progen2", "input", "run_progen2_score")

# Score two protein sequences
score_inputs = ProGen2ScoringInput(sequences=["MVLSPADKTN", "MKTLLILAVVAA"])

**Input** — `CausalModelScoringInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequences</code> | <code>list[str]</code> | required | Sequences to score |

In [10]:
# Display config docs
display_api_reference("progen2", "config", "run_progen2_score")

# Configure scoring | see docs above for all fields
score_config = ProGen2ScoringConfig(
    batch_size=2,
    return_logits=False,
    device="cuda",  # Change to "cpu" if no GPU available
)

**Config** — `ProGen2ScoringConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run the model on |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>batch_size</code> | <code>int</code> | <code>1</code> | Sequences per GPU forward pass; raise for throughput, lower if OOM |
| <code>return_logits</code> | <code>bool</code> | <code>False</code> | Include per-position logits in the output (large; disable to save memory) |
| <code>model_checkpoint</code> | <code>Literal['progen2-small', 'progen2-medium', 'progen2-base', 'progen2-oas', 'progen2-large', 'progen2-BFD90', 'progen2-xlarge']</code> | <code>'progen2-large'</code> | ProGen2 weights variant |
| <code>local_path</code> | <code>str &#124; None</code> | <code>None</code> | Override the default download with a local weights directory |

In [11]:
# Run the scoring function
score_result = run_progen2_score(score_inputs, score_config)

Running run_progen2_score [00:00]

In [12]:
# Display output docs
display_api_reference("progen2", "output", "run_progen2_score")

# Show scoring results for each input sequence
for i, seq in enumerate(score_inputs.sequences):
    score = score_result.scores[i]
    print(f"Sequence {i + 1}: {seq}")
    print(f"    Log-likelihood:     {score.log_likelihood:.3f}")
    print(f"    Avg log-likelihood: {score.avg_log_likelihood:.3f}")
    print(f"    Perplexity:         {score.perplexity:.3f}")

**Output** — `CausalModelScoringOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>scores</code> | <code>list[CausalModelScoringMetrics]</code> | required | List of scoring outputs, one per input sequence |

**Metrics** — `CausalModelScoringMetrics` (one set per `scores` item)

| Metric | Type | Range | Unit | Description |
|--------|------|-------|------|-------------|
| <code>log_likelihood</code> | <code>float</code> | <code>&lt;= 0</code> |  |  |
| <code>avg_log_likelihood</code> | <code>float</code> | <code>&lt;= 0</code> |  |  |
| <code>perplexity</code> **(primary)** | <code>float</code> | <code>&gt;= 1</code> |  |  |

Sequence 1: MVLSPADKTN
    Log-likelihood:     -26.635
    Avg log-likelihood: -2.664
    Perplexity:         14.346
Sequence 2: MKTLLILAVVAA
    Log-likelihood:     -24.403
    Avg log-likelihood: -2.034
    Perplexity:         7.641


## Export Results

Generated sequences and scoring metrics can be saved to standard file formats for downstream use. Sampling results support FASTA, TXT, and JSON export. Scoring results support CSV and JSON export.

In [13]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

sample_result.export(name="progen2_sequences", export_path=output_dir, file_format="fasta")
print(f"Generated sequences exported to {output_dir / 'progen2_sequences.fasta'}")

score_result.export(name="progen2_scores", export_path=output_dir, file_format="csv")
print(f"Scores exported to {output_dir / 'progen2_scores.csv'}")

Generated sequences exported to example_output/progen2_sequences.fasta
Scores exported to example_output/progen2_scores.csv
